In [1]:
import os
import gc
import sys
import time
import math
import json
import random
import numpy as np
import pandas as pd
import typing as tp
from tqdm.auto import tqdm

import seaborn as sns
import matplotlib.pyplot as plt

import ast
from pathlib import Path
from itertools import islice
from collections import Counter, defaultdict

import wave
import soundfile

import librosa
import soundfile as sf
from IPython.display import Audio, display

from scipy.sparse import coo_matrix

import shutil

import glob
import warnings
warnings.filterwarnings("ignore")

In [2]:
class CFG():
    SEED          = 42
    N_FOLDS       = 5
    FOLDS_LIST    = [1]
    base_dir      = "/kaggle/input/competitions/birdclef-2026"
    data_dir      = "/kaggle/input/datasets/tatsuyayamamoto/bird-2026-5fold-df-and-ss/data"
    # Librosa
    FS            = 32_000
    N_FFT         = 1_024
    HOP_LEN       = 512
    N_MELS        = 128
    FMIN          = 50
    FMAX          = 14_000
    DURATION      = 5
    # other
    debug         = False
cfg = CFG()
print(f"Debug : {cfg.debug}")


Debug : False


### SEED Everything

In [3]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    print(f"SEED is {seed}")
    
seed_everything(cfg.SEED)

SEED is 42


### Make Directry

In [4]:
DATA = "./data"
if not os.path.exists(DATA):
    os.makedirs(DATA)

AUDIO = "./audio"
if not os.path.exists(AUDIO):
    os.makedirs(AUDIO)

In [5]:
def safe_literal_eval(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []

def load_df(path):
    df = pd.read_csv(path)
    df["labels"] = df["labels"].apply(safe_literal_eval)
    return df

In [6]:
train = load_df(os.path.join(cfg.data_dir, "train.csv"))
ss_df = load_df(os.path.join(cfg.data_dir, "ss.csv"))
print(f"Train Shape: {train.shape}")
print(f"SS    Shape: {ss_df.shape}")

Train Shape: (35379, 12)
SS    Shape: (425, 10)


In [7]:
train["fold"].value_counts().sort_index()

fold
0    6989
1    7155
2    7028
3    7125
4    7082
Name: count, dtype: int64

In [8]:
for f in range(cfg.N_FOLDS):
    print(train.loc[train["fold"]==f, "group_id"].nunique())

6963
7094
7011
7072
7043


### Split Fold df

In [9]:
fold_df = train[train["fold"]==cfg.FOLDS_LIST[0]].reset_index(drop=True)
print(f"Fold {cfg.FOLDS_LIST[0]} df : {len(fold_df)}")

Fold 1 df : 7155


In [10]:
fold_df["group_id"].nunique()

7094

In [11]:
print(f'300 >= audio      : {fold_df[fold_df["duration"]<300].shape[0]}')
print(f'300 =< audio < 600: {fold_df[(fold_df["duration"]>=300)&(fold_df["duration"]<600)].shape[0]}')
print(f'600 =< audio      : {fold_df[fold_df["duration"]>=600].shape[0]}')

300 >= audio      : 7053
300 =< audio < 600: 29
600 =< audio      : 73


In [12]:
fold_df[fold_df["duration"]>=600].groupby("group_id")["audio_id"].count()

group_id
baymac/XC705946         3
bkcdon/XC703631         4
chacha1/XC417966        3
coffal1/XC979701       18
grekis/XC936081         4
houspa/XC807192        11
rufnig1/iNat1166566     3
rufnig1/iNat1166568     4
thlwre1/XC966000        3
undtin1/XC1047952      13
wesfie1/XC529369        3
yecpar/XC862355         4
Name: audio_id, dtype: int64

In [13]:
display(fold_df.loc[fold_df["duration"]<300, ["audio_id", "duration"]][:3])
display(fold_df.loc[(fold_df["duration"]>=300)&(fold_df["duration"]<600), ["audio_id", "duration"]][:3])
display(fold_df.loc[fold_df["duration"]>=600, ["audio_id", "duration"]][:3])

,audio_id,duration
0,1161364/iNat818781,61.2
1,1161364/iNat840159,7.5
2,116570/iNat1460166,7.8


,audio_id,duration
7053,baffal1/XC530447,461.8
7054,baffal1/XC367718,424.6
7055,batbel1/XC155557,570.0


,audio_id,duration
7082,baymac/XC705946_0-300,663.3
7083,baymac/XC705946_240-540,663.3
7084,baymac/XC705946_363-663,663.3


In [14]:
def split_segment_id(audio_id, duration):
    if "_" in audio_id and "-" in audio_id.split("_")[-1]:
        base, seg  = audio_id.rsplit("_", 1)
        start, end = map(float, seg.split("-"))
    else:
        base = audio_id
        start = 0.0
        end = duration
    return base, start, end

In [15]:
fold_df[["audio_id_base", "start", "end"]] = fold_df[["audio_id", "duration"]].apply(
    lambda x: pd.Series(split_segment_id(x[0], x[1])),
    axis=1
)

In [16]:
COLS = ["audio_id", "duration", "start", "end"]
display(fold_df.loc[fold_df["duration"]<300, COLS][:3])
display(fold_df.loc[(fold_df["duration"]>=300)&(fold_df["duration"]<600), COLS][:3])
display(fold_df.loc[fold_df["duration"]>=600, COLS][:3])

,audio_id,duration,start,end
0,1161364/iNat818781,61.2,0.0,61.2
1,1161364/iNat840159,7.5,0.0,7.5
2,116570/iNat1460166,7.8,0.0,7.8


,audio_id,duration,start,end
7053,baffal1/XC530447,461.8,0.0,461.8
7054,baffal1/XC367718,424.6,0.0,424.6
7055,batbel1/XC155557,570.0,0.0,570.0


,audio_id,duration,start,end
7082,baymac/XC705946_0-300,663.3,0.0,300.0
7083,baymac/XC705946_240-540,663.3,240.0,540.0
7084,baymac/XC705946_363-663,663.3,363.0,663.0


In [17]:
print(fold_df.shape)

(7155, 15)


In [18]:
fold_df["audio_id"] = fold_df["audio_id_base"]
fold_df = fold_df.drop(columns=["audio_id_base"])

In [19]:
fold_df = fold_df.drop_duplicates(
    subset=["audio_id", "start", "end"]
).reset_index(drop=True)

In [20]:
print(fold_df.shape)

(7155, 14)


In [21]:
COLS = ["audio_id", "duration", "start", "end"]
display(fold_df.loc[fold_df["duration"]<300, COLS][:3])
display(fold_df.loc[(fold_df["duration"]>=300)&(fold_df["duration"]<600), COLS][:3])
display(fold_df.loc[fold_df["duration"]>=600, COLS][:3])

,audio_id,duration,start,end
0,1161364/iNat818781,61.2,0.0,61.2
1,1161364/iNat840159,7.5,0.0,7.5
2,116570/iNat1460166,7.8,0.0,7.8


,audio_id,duration,start,end
7053,baffal1/XC530447,461.8,0.0,461.8
7054,baffal1/XC367718,424.6,0.0,424.6
7055,batbel1/XC155557,570.0,0.0,570.0


,audio_id,duration,start,end
7082,baymac/XC705946,663.3,0.0,300.0
7083,baymac/XC705946,663.3,240.0,540.0
7084,baymac/XC705946,663.3,363.0,663.0


In [22]:
fold_df["audio_id"] = fold_df["audio_id"].apply(lambda x: x.replace("/", "_"))
print(fold_df["audio_id"][:3])

0    1161364_iNat818781
1    1161364_iNat840159
2    116570_iNat1460166
Name: audio_id, dtype: object


### Fold DF Save

In [23]:
fold_df.to_csv(os.path.join(DATA, F"fold_{cfg.FOLDS_LIST[0]}.csv"), index=False)

### Audio Save

In [24]:
unique_df = fold_df[["audio_id", "filename"]].drop_duplicates()
len(unique_df)

7094

In [25]:
skip_ids   = []
file_type  = "train_audio"
AUDIO_FOLD = os.path.join(AUDIO, f"fold_{cfg.FOLDS_LIST[0]}")
os.makedirs(AUDIO_FOLD, exist_ok=True)

for _, row in tqdm(unique_df.iterrows(), total=len(unique_df), desc="Audio save"):
    filename = row.filename
    audio_id = row.audio_id

    wav, sr = sf.read(os.path.join(cfg.base_dir, file_type, filename), dtype="float32")
    save_path = os.path.join(AUDIO_FOLD, f"{audio_id}.wav")

    try:
        if wav is None or len(wav) == 0:
            print("EMPTY:", audio_id)
            continue

        if np.isnan(wav).any():
            print("NaN:", audio_id)
            continue

        wav = np.asarray(wav, dtype=np.float32)

        # 👇 subtype変更（超重要）
        sf.write(save_path, wav, samplerate=sr, subtype="PCM_16")
        # FLOATは重い＆不安定
	    # PCM_16は軽くて安定（音声系の標準）

    except Exception as e:
        print("SKIP:", audio_id)
        print("shape:", wav.shape if wav is not None else None)
        print("dtype:", wav.dtype if wav is not None else None)
        print(e)
        skip_ids.append(audio_id)

Audio save:   0%|          | 0/7094 [00:00<?, ?it/s]

In [26]:
print(len(unique_df))
print(len(glob.glob(os.path.join(AUDIO_FOLD, "**/*.wav"), recursive=True)))

7094
7094


In [27]:
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Free: {free/1e9:.2f} GB")

Free: 4.59 GB
